#### Preparing Data

In [66]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [67]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/ALLMAT_200k_training_data.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [68]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

Number of rows before cleaning: 200000


In [69]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()

# Applying log transformations
df['capacity'] = np.log(df['capacity'])
df['ads_time'] = np.log(df['ads_time'])
df['C_o'] = np.log(df['C_o'])
df['dp'] = np.log(df['dp'])
df['qm1'] = np.log(df['qm1'])
df['kl1'] = np.log(df['kl1'])
df['sorbent_cost_per_kg'] = np.log(df['sorbent_cost_per_kg'])
df['total_LCP'] = np.log(df['total_LCP'])
df['emissions'] = np.log(df['emissions'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

capacity                 2.304194
diameter                 0.955110
column_height            1.000010
porosity                 0.350001
ads_time                 3.401257
C_o                     -6.815012
rho_s                  500.004000
dp                      -8.111725
qm1                     -6.775754
kl1                      0.003334
sorbent_cost_per_kg      0.003125
total_LCP                9.997128
emissions                5.171745
PFAS_eff               -46.051702
PFAS_ads                 0.000022
dtype: float64
capacity                  8.517189
diameter                  4.000000
column_height             3.999990
porosity                  0.499998
ads_time                  7.509330
C_o                       2.302583
rho_s                  1999.980000
dp                       -5.991469
qm1                       2.302564
kl1                       9.210334
sorbent_cost_per_kg       7.600902
total_LCP                29.519682
emissions                19.495519
PFAS_eff            

In [70]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

Number of rows after cleaning: 200000


In [71]:
# separate the data into inputs and outputs
inputs = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    ]

outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff"
]

dfin = df[inputs]
dfout = df[outputs]

In [72]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [73]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [74]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [75]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [76]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [77]:
# Optimisation Data
obj_scaler = 1e-6

# process parameters
feed_conc_val = 0.05 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 1000

# materials properties
density_val = 501 # kg/m3
porosity_val = 0.4
dp_val = 0.002 # m
qm1_val = 0.5 # mol/kg
kl1_val = 10 # m3/mol
sorbent_cost_val = 2 # $/kg

# loading saved nn
nn_PFAS_model = keras.models.load_model("ALLMAT_relu_mse.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, density=density_val, porosity=porosity_val, dp=dp_val, qm1=qm1_val, kl1=kl1_val, sorbent_cost=sorbent_cost_val):

    max_capacity_per_bed = 120 # m3/h
    N_beds = int(np.ceil(capacity/max_capacity_per_bed))
    Q_per_column = capacity/N_beds

    optimization_model = pyo.ConcreteModel()
    
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.dp = pyo.Var(initialize=(unscaled_lb[7]+unscaled_ub[7])/2, bounds=(unscaled_lb[7], unscaled_ub[7]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    optimization_model.nn = OmltBlock()
    formulated_problem = ReluBigMFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    optimization_model.nn.inputs[0].fix(pyo.log(capacity))
    optimization_model.nn.inputs[5].fix(pyo.log(feed_conc))
    optimization_model.nn.inputs[6].fix(density)
    optimization_model.nn.inputs[8].fix(pyo.log(qm1))
    optimization_model.nn.inputs[9].fix(pyo.log(kl1))
    optimization_model.nn.inputs[10].fix(pyo.log(sorbent_cost))

    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_dp = pyo.Constraint(expr= optimization_model.dp == optimization_model.nn.inputs[7])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    optimization_model.min_velocity = pyo.Constraint(expr= 4*Q_per_column >= 2*(np.pi*optimization_model.diameter**2))
    optimization_model.max_velocity = pyo.Constraint(expr= 4*Q_per_column <= 15*(np.pi*optimization_model.diameter**2))

    if eps_emissions is not None:
        optimization_model.emissions_con = pyo.Constraint(expr= optimization_model.emissions <= eps_emissions)
    
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr=(optimization_model.total_LCP), sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr=(optimization_model.emissions), sense=pyo.minimize)

    options = {
    'MaxTime': 500,
    'epsr': 0.05,
    'epsa': 1e-4,
    'threads': 16,
    'prlevel': 1,
    'dolocal': 1
    }

    # print("# of variables: ", optimization_model.nvariables())
    # print("# of constraints: ", optimization_model.nconstraints())
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=False, options=options)
    
    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        # print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        # print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        # print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            N_beds,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            np.exp(get_val(optimization_model.dp)),
            np.exp(get_val(optimization_model.ads_time)),
            feed_conc,
            density,
            qm1,
            kl1,
            sorbent_cost,
            np.exp(get_val(optimization_model.total_LCP)),
            np.exp(get_val(optimization_model.emissions)),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [78]:
import itertools
import time

def run_affinity_cost_sweep(capacity_val, qm1_fixed, prod_bounds, cost_bounds, qm1_bounds, kl1_bounds, grid_size=15):
    """
    Runs a 2D parametric sweep of (q_max * K_L) vs Sorbent Cost for a fixed capacity.
    q_max is held constant, but if K_L hits a physical boundary, q_max flexes to maintain the product.
    """
    prod_min, prod_max = prod_bounds
    cost_min, cost_max = cost_bounds
    qm1_min, qm1_max = qm1_bounds
    kl1_min, kl1_max = kl1_bounds
    
    # Generate values for the sweep
    prod_values = np.linspace(prod_min, prod_max, grid_size)
    cost_values = np.linspace(cost_min, cost_max, grid_size)
    
    run_grid = list(itertools.product(prod_values, cost_values))
    total_runs = len(run_grid)
    
    print(f"\n========================================================")
    print(f"STARTING SWEEP: Capacity = {capacity_val} m3/h | Target Fixed q_max = {qm1_fixed:.2f}")
    print(f"Grid: {grid_size}x{grid_size} ({total_runs} total optimization runs)")
    print(f"========================================================\n")
    
    results_data = []
    start_time = time.time()

    for idx, (target_prod, cost_val) in enumerate(run_grid):
        
        # 1. Attempt to hit product using the fixed Qmax
        q_val = qm1_fixed
        kl_val = target_prod / qm1_fixed
        
        # 2. BOUNDARY CHECK: If KL is out of bounds, clamp it and flex Qmax
        if kl_val > kl1_max:
            kl_val = kl1_max
            q_val = target_prod / kl_val
        elif kl_val < kl1_min:
            kl_val = kl1_min
            q_val = target_prod / kl_val
            
        # 3. SECONDARY CHECK: Did flexing Qmax push it out of bounds?
        if q_val > qm1_max or q_val < qm1_min:
            print(f"Run {idx+1}/{total_runs} | Qmax*KL: {target_prod:.2f} | ⚠️ SKIPPED (Requires physical params outside known bounds)")
            
            failed_res = (
                capacity_val, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 
                np.nan, np.nan, q_val, kl_val, cost_val, np.nan, np.nan, np.nan, 
                "Out of Bounds", target_prod
            )
            results_data.append(failed_res)
            continue # Skip the solver completely to save time
            
        print(f"Run {idx+1}/{total_runs} | Qmax*KL: {target_prod:.2f} | Cost: ${cost_val:.2f}/kg | (q_max: {q_val:.2f}, K_L: {kl_val:.4f})")
        
        try:
            res = solve_model(
                objective='cost',
                eps_emissions=None,
                capacity=capacity_val,
                sorbent_cost=cost_val,
                qm1=q_val,
                kl1=kl_val
            )
            
            res_list = list(res)
            res_list.append(target_prod) 
            results_data.append(res_list)
            
        except Exception as e:
            print(f"  -> ⚠️ Solver crashed. Error: {e}")
            failed_res = (
                capacity_val, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 
                np.nan, np.nan, q_val, kl_val, cost_val, np.nan, np.nan, np.nan, 
                "Solver Crashed", target_prod
            )
            results_data.append(failed_res)

    columns = [
        "capacity", "N_beds", "diameter", "column_height", "porosity", "dp", 
        "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
        "total_LCP", "emissions", "PFAS_eff_conc", "status", "qmax_kl_product"
    ]
    df_results = pd.DataFrame(results_data, columns=columns)
    
    excel_filename = f"optimisation_QmaxKL_vs_Cost_Cap{capacity_val}.xlsx"
    df_results.to_excel(excel_filename, index=False)
    
    elapsed_mins = (time.time() - start_time) / 60
    print(f"\n✅ Sweep complete in {elapsed_mins:.1f} mins. Saved to {excel_filename}")
    return df_results

# ==========================================
# MASTER EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    
    # 1. Define ALL physical bounds based on your underlying data
    qm1_min = df1['qm1'].min()
    qm1_max = df1['qm1'].max()
    kl1_min = df1['kl1'].min()
    kl1_max = df1['kl1'].max()
    
    prod_min = (df1['qm1'] * df1['kl1']).min()
    prod_max = (df1['qm1'] * df1['kl1']).max()
    sorbent_cost_min = df1['sorbent_cost_per_kg'].min()
    sorbent_cost_max = df1['sorbent_cost_per_kg'].max()
    
    QM1_BOUNDS = (qm1_min, qm1_max)
    KL1_BOUNDS = (kl1_min, kl1_max)
    QM1_BASELINE = df1['qm1'].mean()                   
    PROD_BOUNDS = (prod_min, prod_max)              
    COST_BOUNDS = (sorbent_cost_min, sorbent_cost_max)              
    
    capacities_to_test = [100, 1000, 2000] # m3/h
    
    for cap in capacities_to_test:
        run_affinity_cost_sweep(
            capacity_val=cap,
            qm1_fixed=QM1_BASELINE,
            prod_bounds=PROD_BOUNDS,
            cost_bounds=COST_BOUNDS,
            qm1_bounds=QM1_BOUNDS,      # Passed to function
            kl1_bounds=KL1_BOUNDS,      # Passed to function
            grid_size=10
        )


STARTING SWEEP: Capacity = 100 m3/h | Target Fixed q_max = 5.01
Grid: 10x10 (100 total optimization runs)

Run 1/100 | Qmax*KL: 0.11 | Cost: $1.00/kg | (q_max: 0.10, K_L: 1.0033)
Run 2/100 | Qmax*KL: 0.11 | Cost: $223.11/kg | (q_max: 0.10, K_L: 1.0033)
Run 3/100 | Qmax*KL: 0.11 | Cost: $445.22/kg | (q_max: 0.10, K_L: 1.0033)
Run 4/100 | Qmax*KL: 0.11 | Cost: $667.34/kg | (q_max: 0.10, K_L: 1.0033)
Run 5/100 | Qmax*KL: 0.11 | Cost: $889.45/kg | (q_max: 0.10, K_L: 1.0033)
Run 6/100 | Qmax*KL: 0.11 | Cost: $1111.56/kg | (q_max: 0.10, K_L: 1.0033)
Run 7/100 | Qmax*KL: 0.11 | Cost: $1333.67/kg | (q_max: 0.10, K_L: 1.0033)
Run 8/100 | Qmax*KL: 0.11 | Cost: $1555.78/kg | (q_max: 0.10, K_L: 1.0033)
Run 9/100 | Qmax*KL: 0.11 | Cost: $1777.89/kg | (q_max: 0.10, K_L: 1.0033)
Run 10/100 | Qmax*KL: 0.11 | Cost: $2000.00/kg | (q_max: 0.10, K_L: 1.0033)
Run 11/100 | Qmax*KL: 11049.26 | Cost: $1.00/kg | (q_max: 5.01, K_L: 2206.6304)
Run 12/100 | Qmax*KL: 11049.26 | Cost: $223.11/kg | (q_max: 5.01, K_